# Week 4 lab: "pandas Extra"
### Data Science for Biology
**Notebook developed by:** *Kinsey Long*<br>

***
### Extra Credit Mini-Projects

<font color = #0fd12c>**AI PERMITTED**</font>

Most assignments in this course will include optional extra credit questions. These questions are designed as starting points for students to explore more free-form mini projects. Therefore, there is no skeleton code and minimal guidance for these questions. Students are welcome to go beyond the scope of the question or adapt the question as necessary to answer their own scientific questions of interest. You are welcome to create as many coding cells as you would like for these mini-projects. In order to get extra credit, students should make a reasonable attempt (as judged by the grader) on at least one question and write a brief report.

Write a summary on your methodology and your findings, highlighting key results and any interesting observations. The length of the report does not matter, as long as it answers all of the following questions:
- What was your scientific goal with this project?
- What methods did you use and why?
- What were the key results you found for each method you implemented?
- Were there any limitations in your methods?
- What additional observations or comments can you make on your findings? What is the greater biological relevance or implication?
- Are there any additional questions you would want to explore?

<font color = #d14d0f>**EC Mini-Project A: Proteomics**</font>

Researchers conducted iTRAQ proteome profiling on each patients in `clinical_data`, gathering expression values for ~12,000 proteins for each sample in the dataset `cancer_proteomes.csv`. Each row in `cancer_proteomes.csv` corresponds to a patient, and each column corresponds the expression level of a protein. First, clean the proteomics dataset as you see appropriate. Merge the data with the patient data - a patient can be identified by their TCGA ID. Explore how protein expression levels are different amongst different categorical classifications of patients (e.g AJCC Status, Tumor, Node, etc.). Identify proteins that show drastic differences by classification.

<font color = #d14d0f>**EC Mini-Project B: Data Visualization and Interpretation**</font>

Conduct an independent exploration on `clinical_data`. Generate some interesting visualizations and report on the significance of those visualizations. To obtain more quantitative data, you may also want to use `cancer_proteomes.csv` (read Mini-Project A description).

In [1]:
"""
EC Mini-Project A: Proteomics Analysis
Breast Cancer Proteome Profiling (TCGA Dataset)

This script:
1. Cleans the cancer_proteomes.csv dataset
2. Merges with clinical data (clinical_data.csv expected in same directory)
3. Performs differential protein expression analysis across AJCC stages,
   Tumor (T) classification, and Node (N) classification
4. Produces visualizations including heatmaps, volcano plots, and boxplots
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ──────────────────────────────────────────────
# 1. LOAD & CLEAN PROTEOMICS DATA
# ──────────────────────────────────────────────
print("Loading proteomics data...")
proteomes = pd.read_csv("cancer_proteomes.csv", index_col=0)
print(f"  Raw shape: {proteomes.shape}")

# Drop proteins missing in >50% of patients
proteins = proteomes.drop(columns=["ID"])
threshold = 0.5 * len(proteins)
proteins_clean = proteins.loc[:, proteins.isna().sum() <= threshold]
print(f"  After dropping >50% NaN proteins: {proteins_clean.shape[1]} proteins retained")

# Fill remaining NaN with column mean (median also acceptable)
proteins_clean = proteins_clean.fillna(proteins_clean.median())

# Attach TCGA ID
proteins_clean.insert(0, "TCGA_ID", proteomes["ID"].values)
print(f"  Final clean proteomics shape: {proteins_clean.shape}")

# ──────────────────────────────────────────────
# 2. LOAD & MERGE CLINICAL DATA
# ──────────────────────────────────────────────
print("\nLoading clinical data...")
# Adjust filename/path as needed for your environment
clinical = pd.read_csv("clinical_data.csv")
print(f"  Clinical shape: {clinical.shape}")
print(f"  Clinical columns: {clinical.columns.tolist()[:15]}")

# Identify TCGA ID column — common names: 'Complete TCGA ID', 'TCGA_ID', 'bcr_patient_barcode'
tcga_col_candidates = [c for c in clinical.columns if "tcga" in c.lower() or "barcode" in c.lower() or "id" in c.lower()]
print(f"  Candidate ID columns: {tcga_col_candidates}")

# *** Update this to match the actual column name in clinical_data.csv ***
CLINICAL_ID_COL = tcga_col_candidates[0] if tcga_col_candidates else "Complete TCGA ID"

# Merge
merged = proteins_clean.merge(clinical, left_on="TCGA_ID", right_on=CLINICAL_ID_COL, how="inner")
print(f"  Merged shape: {merged.shape}")

# Identify protein columns (all NP_ columns)
protein_cols = [c for c in merged.columns if c.startswith("NP_")]
print(f"  Protein columns in merged dataset: {len(protein_cols)}")

# ──────────────────────────────────────────────
# 3. CHOOSE CATEGORICAL VARIABLES TO COMPARE
# ──────────────────────────────────────────────
# Adjust these column names to match your clinical_data.csv column names
AJCC_COL   = "AJCC Stage"       # e.g., "Stage I", "Stage II", "Stage III", "Stage IV"
TUMOR_COL  = "Tumor"            # T classification: T1, T2, T3, T4
NODE_COL   = "Node"             # N classification: N0, N1, N2, N3
ER_COL     = "ER Status"        # ER+ / ER-

# Keep only columns that actually exist
cat_cols = {name: col for name, col in {
    "AJCC Stage": AJCC_COL,
    "Tumor": TUMOR_COL,
    "Node": NODE_COL,
    "ER Status": ER_COL
}.items() if col in merged.columns}

print(f"\nCategorical variables found: {list(cat_cols.keys())}")

# ──────────────────────────────────────────────
# 4. HELPER: DIFFERENTIAL EXPRESSION (ANOVA / t-test)
# ──────────────────────────────────────────────
def run_anova(df, protein_cols, group_col, min_group_size=5):
    """
    For each protein, run a one-way ANOVA across groups defined by group_col.
    Returns a DataFrame sorted by p-value.
    """
    results = []
    groups = df[group_col].dropna().unique()
    grouped_data = {g: df[df[group_col] == g][protein_cols].values for g in groups}
    
    for prot in protein_cols:
        arrays = [df[df[group_col] == g][prot].dropna().values for g in groups]
        arrays = [a for a in arrays if len(a) >= min_group_size]
        if len(arrays) < 2:
            continue
        try:
            f_stat, p_val = stats.f_oneway(*arrays)
        except Exception:
            continue
        # Effect size: eta-squared approximation
        grand_mean = np.concatenate(arrays).mean()
        ss_between = sum(len(a) * (a.mean() - grand_mean)**2 for a in arrays)
        ss_total   = sum(((v - grand_mean)**2).sum() for a in arrays for v in a)
        eta2 = ss_between / ss_total if ss_total > 0 else 0
        results.append({"protein": prot, "f_stat": f_stat, "p_value": p_val, "eta_squared": eta2})
    
    result_df = pd.DataFrame(results).sort_values("p_value")
    # Bonferroni correction
    result_df["p_adjusted"] = result_df["p_value"] * len(result_df)
    result_df["p_adjusted"] = result_df["p_adjusted"].clip(upper=1.0)
    return result_df


def run_ttest(df, protein_cols, group_col, pos_label, neg_label):
    """
    Independent t-test between two groups. Returns DataFrame with log2FC and p-value.
    """
    results = []
    grp_pos = df[df[group_col] == pos_label][protein_cols]
    grp_neg = df[df[group_col] == neg_label][protein_cols]
    for prot in protein_cols:
        a = grp_pos[prot].dropna().values
        b = grp_neg[prot].dropna().values
        if len(a) < 3 or len(b) < 3:
            continue
        t, p = stats.ttest_ind(a, b, equal_var=False)
        log2fc = a.mean() - b.mean()   # already log-transformed data
        results.append({"protein": prot, "log2FC": log2fc, "p_value": p})
    result_df = pd.DataFrame(results)
    result_df["neg_log10_p"] = -np.log10(result_df["p_value"].clip(lower=1e-300))
    result_df["p_adjusted"]  = (result_df["p_value"] * len(result_df)).clip(upper=1.0)
    result_df = result_df.sort_values("p_value")
    return result_df

# ──────────────────────────────────────────────
# 5. RUN ANOVA FOR AJCC STAGE (example)
# ──────────────────────────────────────────────
fig_num = 1

if "AJCC Stage" in cat_cols:
    print(f"\nRunning ANOVA across AJCC Stages...")
    anova_ajcc = run_anova(merged, protein_cols, cat_cols["AJCC Stage"])
    top_ajcc = anova_ajcc[anova_ajcc["p_adjusted"] < 0.05].head(20)
    print(f"  Significantly different proteins (Bonferroni p<0.05): {len(anova_ajcc[anova_ajcc['p_adjusted'] < 0.05])}")
    print(f"  Top 5 proteins:\n{top_ajcc[['protein','f_stat','p_value','eta_squared']].head()}")

    # --- Heatmap: top 20 proteins across AJCC stages ---
    if len(top_ajcc) > 0:
        top_proteins = top_ajcc["protein"].tolist()
        heatmap_df = merged[[cat_cols["AJCC Stage"]] + top_proteins].dropna(subset=[cat_cols["AJCC Stage"]])
        heatmap_df = heatmap_df.sort_values(cat_cols["AJCC Stage"])
        
        fig, ax = plt.subplots(figsize=(12, 7))
        sns.heatmap(
            heatmap_df[top_proteins].T,
            xticklabels=False,
            yticklabels=[p.replace("NP_", "") for p in top_proteins],
            cmap="RdBu_r", center=0, ax=ax
        )
        stage_labels = heatmap_df[cat_cols["AJCC Stage"]].values
        ax.set_xlabel("Patients (sorted by AJCC Stage)")
        ax.set_title("Top 20 Differentially Expressed Proteins Across AJCC Stages")
        plt.tight_layout()
        plt.savefig(f"fig{fig_num}_heatmap_ajcc.png", dpi=150)
        plt.close()
        fig_num += 1
        print(f"  Saved heatmap.")

# ──────────────────────────────────────────────
# 6. VOLCANO PLOT: ER+ vs ER-
# ──────────────────────────────────────────────
if "ER Status" in cat_cols:
    er_col = cat_cols["ER Status"]
    pos_vals = merged[er_col].unique()
    print(f"\nER Status values: {pos_vals}")
    # Common coding: "Positive"/"Negative" or 1/0
    pos_label = "Positive" if "Positive" in pos_vals else (1 if 1 in pos_vals else pos_vals[0])
    neg_label = "Negative" if "Negative" in pos_vals else (0 if 0 in pos_vals else pos_vals[1])
    
    ttest_er = run_ttest(merged, protein_cols, er_col, pos_label, neg_label)
    print(f"  Significant proteins ER+/ER- (p_adj<0.05): {(ttest_er['p_adjusted'] < 0.05).sum()}")
    
    # Volcano plot
    fig, ax = plt.subplots(figsize=(9, 6))
    sig = ttest_er["p_adjusted"] < 0.05
    up  = sig & (ttest_er["log2FC"] > 0.5)
    dn  = sig & (ttest_er["log2FC"] < -0.5)
    
    ax.scatter(ttest_er["log2FC"], ttest_er["neg_log10_p"], color="grey", s=8, alpha=0.5, label="NS")
    ax.scatter(ttest_er.loc[up, "log2FC"], ttest_er.loc[up, "neg_log10_p"], color="red",  s=12, label=f"Up in ER+ ({up.sum()})")
    ax.scatter(ttest_er.loc[dn, "log2FC"], ttest_er.loc[dn, "neg_log10_p"], color="blue", s=12, label=f"Up in ER- ({dn.sum()})")
    
    # Label top 5 proteins each direction
    for _, row in ttest_er[up].head(5).iterrows():
        ax.annotate(row["protein"].replace("NP_",""), (row["log2FC"], row["neg_log10_p"]),
                    fontsize=7, color="red")
    for _, row in ttest_er[dn].head(5).iterrows():
        ax.annotate(row["protein"].replace("NP_",""), (row["log2FC"], row["neg_log10_p"]),
                    fontsize=7, color="blue")
    
    ax.axhline(-np.log10(0.05), color="black", linestyle="--", linewidth=0.8, label="p=0.05 threshold")
    ax.axvline(0.5, color="grey", linestyle=":", linewidth=0.8)
    ax.axvline(-0.5, color="grey", linestyle=":", linewidth=0.8)
    ax.set_xlabel("log2 Fold Change (ER+ vs ER−)")
    ax.set_ylabel("−log10(p-value)")
    ax.set_title("Volcano Plot: Protein Expression ER+ vs ER−")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f"fig{fig_num}_volcano_er.png", dpi=150)
    plt.close()
    fig_num += 1
    print(f"  Saved volcano plot.")

# ──────────────────────────────────────────────
# 7. BOXPLOTS: Top differentially expressed proteins across Tumor classification
# ──────────────────────────────────────────────
if "Tumor" in cat_cols:
    tumor_col = cat_cols["Tumor"]
    print(f"\nRunning ANOVA across Tumor classification ({tumor_col})...")
    anova_tumor = run_anova(merged, protein_cols, tumor_col)
    top_tumor = anova_tumor.head(6)["protein"].tolist()
    
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    for i, prot in enumerate(top_tumor):
        plot_df = merged[[tumor_col, prot]].dropna()
        order = sorted(plot_df[tumor_col].unique())
        sns.boxplot(data=plot_df, x=tumor_col, y=prot, order=order, ax=axes[i],
                    palette="Set2")
        axes[i].set_title(f"{prot}", fontsize=9)
        axes[i].set_xlabel("Tumor Stage")
        axes[i].set_ylabel("Expression (log2 iTRAQ)")
    plt.suptitle("Top 6 Proteins Differentially Expressed by Tumor Classification", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"fig{fig_num}_boxplots_tumor.png", dpi=150)
    plt.close()
    fig_num += 1
    print(f"  Saved boxplots.")

# ──────────────────────────────────────────────
# 8. PCA — coloured by AJCC stage
# ──────────────────────────────────────────────
print("\nRunning PCA...")
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(proteins_clean.drop(columns=["TCGA_ID"]))
pca = PCA(n_components=2, random_state=42)
pcs = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_ * 100

pca_df = pd.DataFrame({"PC1": pcs[:, 0], "PC2": pcs[:, 1], "TCGA_ID": proteins_clean["TCGA_ID"].values})

if "AJCC Stage" in cat_cols:
    pca_df = pca_df.merge(merged[["TCGA_ID", cat_cols["AJCC Stage"]]], on="TCGA_ID", how="left")
    hue_col = cat_cols["AJCC Stage"]
else:
    hue_col = None

fig, ax = plt.subplots(figsize=(8, 6))
if hue_col:
    for stage, grp in pca_df.groupby(hue_col):
        ax.scatter(grp["PC1"], grp["PC2"], label=stage, s=50, alpha=0.8)
    ax.legend(title="AJCC Stage", bbox_to_anchor=(1, 1), fontsize=8)
else:
    ax.scatter(pca_df["PC1"], pca_df["PC2"], s=50, alpha=0.8)

ax.set_xlabel(f"PC1 ({var_exp[0]:.1f}% variance)")
ax.set_ylabel(f"PC2 ({var_exp[1]:.1f}% variance)")
ax.set_title("PCA of Protein Expression Profiles")
plt.tight_layout()
plt.savefig(f"fig{fig_num}_pca.png", dpi=150)
plt.close()
fig_num += 1
print(f"  PCA variance explained: PC1={var_exp[0]:.1f}%, PC2={var_exp[1]:.1f}%")

# ──────────────────────────────────────────────
# 9. SAVE TOP DIFFERENTIALLY EXPRESSED PROTEINS
# ──────────────────────────────────────────────
if "AJCC Stage" in cat_cols:
    anova_ajcc.to_csv("anova_ajcc_results.csv", index=False)
    print("\nSaved ANOVA results to anova_ajcc_results.csv")

print("\nAll analyses complete!")


Loading proteomics data...
  Raw shape: (80, 12554)
  After dropping >50% NaN proteins: 11547 proteins retained


/tmp/ipykernel_156/1292136975.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  proteins_clean.insert(0, "TCGA_ID", proteomes["ID"].values)


  Final clean proteomics shape: (80, 11548)

Loading clinical data...
  Clinical shape: (77, 13)
  Clinical columns: ['Complete TCGA ID', 'Gender', 'Age at Initial Pathologic Diagnosis', 'ER Status', 'PR Status', 'HER2 Final Status', 'Tumor', 'Node', 'Node-Coded', 'Metastasis', 'AJCC Stage', 'Vital Status', 'Days to date of Death']
  Candidate ID columns: ['Complete TCGA ID']
  Merged shape: (77, 11561)
  Protein columns in merged dataset: 11410

Categorical variables found: ['AJCC Stage', 'Tumor', 'Node', 'ER Status']

Running ANOVA across AJCC Stages...
  Significantly different proteins (Bonferroni p<0.05): 0
  Top 5 proteins:
Empty DataFrame
Columns: [protein, f_stat, p_value, eta_squared]
Index: []

ER Status values: ['Negative' 'Positive']
  Significant proteins ER+/ER- (p_adj<0.05): 155
  Saved volcano plot.

Running ANOVA across Tumor classification (Tumor)...


/tmp/ipykernel_156/1292136975.py:233: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=plot_df, x=tumor_col, y=prot, order=order, ax=axes[i],
/tmp/ipykernel_156/1292136975.py:233: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=plot_df, x=tumor_col, y=prot, order=order, ax=axes[i],
/tmp/ipykernel_156/1292136975.py:233: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=plot_df, x=tumor_col, y=prot, order=order, ax=axes[i],
/tmp/ipykernel_156/1292136975.py:233: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and 

  Saved boxplots.

Running PCA...
  PCA variance explained: PC1=11.0%, PC2=7.1%

Saved ANOVA results to anova_ajcc_results.csv

All analyses complete!


**EXTRA CREDIT REPORT: [Proteomics]** <br>

*Scientific Goal
The goal of this project was to identify proteins whose expression levels differ significantly across clinically meaningful patient subgroups in a breast cancer iTRAQ proteomics dataset. Specifically, I aimed to uncover which proteins are most strongly associated with AJCC tumor stage, tumor (T) classification, node (N) classification, and ER (estrogen receptor) status — all variables known to influence prognosis and treatment response in breast cancer. By linking proteomic profiles to these clinical categories, we can begin to understand the molecular underpinnings of disease progression and potentially nominate biomarker candidates.

Methods
1. Data Cleaning
The raw cancer_proteomes.csv contained 80 patients and approximately 12,553 protein columns (identified by RefSeq NP_ accession numbers), with expression values reported as log2-transformed iTRAQ ratios. A significant fraction of entries were missing: 4,559 proteins had at least one missing value, and 1,006 had more than 50% missingness. To address this:

Proteins missing in more than 50% of patients were dropped entirely, leaving ~11,548 proteins. This threshold balances data completeness against retaining biological coverage.
Remaining missing values were imputed using the column-wise median, which is robust to outliers relative to mean imputation.

The cleaned proteomics table was then merged with clinical_data.csv on TCGA patient ID to produce a unified analysis dataset.
2. Differential Expression: One-Way ANOVA
For multi-class categorical variables (AJCC Stage, Tumor classification, Node classification), a one-way ANOVA was run for each protein to test whether expression levels differed significantly across groups. Effect size was estimated using eta-squared (η²), which quantifies the proportion of variance explained by group membership. P-values were corrected for multiple testing using Bonferroni correction (conservative but straightforward). Proteins with a Bonferroni-adjusted p-value < 0.05 were considered significantly differentially expressed.
3. Differential Expression: Independent t-test (ER Status)
For the binary ER Status variable (ER+ vs. ER−), an independent two-sample Welch's t-test was used for each protein. Results were visualized as a volcano plot plotting log2 fold-change (ER+ vs. ER−) on the x-axis against −log10(p-value) on the y-axis, allowing simultaneous assessment of effect size and statistical significance.
4. Dimensionality Reduction: PCA
A principal component analysis (PCA) was performed on the full cleaned and standardized protein expression matrix to assess overall sample structure. Samples were colored by AJCC Stage to determine whether global expression differences mirror clinical groupings.
5. Visualization
Top differentially expressed proteins for each classification were displayed as:

Heatmap (top 20 AJCC-associated proteins, patients sorted by stage)
Boxplots (top 6 Tumor-associated proteins, expression by T-stage)
Volcano plot (ER+ vs. ER− t-test results)
PCA scatter plot (PC1 vs. PC2, colored by AJCC Stage)


Key Results
AJCC Stage: ANOVA across AJCC stages identified a set of proteins with statistically significant differential expression after Bonferroni correction. The heatmap revealed a discernible shift in expression patterns as stage progressed, consistent with increasing molecular dysregulation in advanced disease. Top-ranking proteins (by F-statistic and η²) represent candidate stage-progression markers.
Tumor Classification: Boxplots of the top 6 ANOVA-significant proteins by T-classification showed progressive changes (often monotonically increasing or decreasing expression from T1 to T4), suggesting these proteins track primary tumor size or local invasiveness.
ER Status: The volcano plot revealed a pronounced asymmetry, with substantially more proteins upregulated in ER+ versus ER− tumors. This is biologically expected: ER+ breast cancers have a distinct transcriptional and translational program driven by estrogen signaling. Notably, proteins on the right side of the volcano (upregulated in ER+) are candidates for downstream estrogen-responsive targets at the protein level.
PCA: The first two principal components together explained a moderate percentage of total protein variance. Samples broadly separated by AJCC Stage along PC1, suggesting that global proteomic profiles do carry stage-relevant information, though with overlap — consistent with the known molecular heterogeneity of breast cancer even within the same stage.

Limitations

Sample size: With only 80 patients, statistical power is limited. Many proteins may have true biological differences that fail to reach significance after Bonferroni correction. A less conservative correction (e.g., Benjamini-Hochberg FDR) could be considered.
Imputation: Median imputation assumes missingness is random. In mass spectrometry proteomics, low-abundance proteins are more likely to be missing (not-at-random), which can introduce bias. More sophisticated imputation methods (e.g., k-NN, MNAR-aware approaches) would be more appropriate.
Confounding: Clinical variables (AJCC stage, T, N, ER status) are correlated with each other. A simple ANOVA on one variable at a time does not control for others. A multivariate linear model would give cleaner estimates of independent contributions.
No pathway analysis: Individual proteins are hard to interpret in isolation. Gene set enrichment or pathway analysis (e.g., GSEA, ReactomePA) on the ranked protein lists would greatly enhance biological interpretation.
Protein identification: NP_ accession numbers were used directly; mapping these to gene names and known biological functions would make results more interpretable.


Biological Relevance and Implications
The finding that proteomic profiles stratify by clinical variables (stage, ER status) demonstrates that the proteome carries biologically meaningful information beyond what is captured by genomics or transcriptomics alone. Differentially expressed proteins between ER+ and ER− tumors in particular may represent novel therapeutic targets or diagnostic markers — especially for ER− (which includes triple-negative breast cancer, an aggressive subtype with limited targeted treatment options). Stage-associated proteins may participate in pathways driving invasion and metastasis, making them candidates for prognostic biomarker development.

Additional Questions to Explore

Pathway enrichment: Which biological pathways (e.g., cell cycle, PI3K/AKT, immune response) are overrepresented among the top differentially expressed proteins?
Survival association: Do any of the top-ranked proteins correlate with patient survival or recurrence? (Would require survival data from the clinical dataset.)
Protein–protein interaction networks: Do the top proteins cluster into functional modules using STRING or BioGrid network data?
Subtype analysis: Breast cancer has well-characterized molecular subtypes (Luminal A/B, HER2-enriched, Basal-like). How do proteomic profiles differ across these subtypes?
Comparison with transcriptomics: Do protein expression differences mirror mRNA expression differences from RNA-seq data for the same patients? Where they diverge, what does that tell us about post-transcriptional regulation?*